# nb_mdf_40_operations_monitor - run the platform, not just build it

Everything an operations team needs is already in the control tables. This notebook answers the daily questions:

1. Did everything load? (latest status per object and layer)
2. What failed and why? (with the exact rerun command)
3. Are we within SLA? (data freshness vs `sla_hours`)
4. How good is the data? (technical vs business DQ scorecard)
5. What is sitting in quarantine, and what resolved itself?
6. Where are the watermarks? (what the next incremental run will pick up)

Turn these queries into a Power BI report on the SQL analytics endpoint, or alert on them with Activator.

In [ ]:
%run nb_mdf_utils

### 1. Latest status per object and layer

In [ ]:
display(spark.sql(f"""
WITH r AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY object_id, layer ORDER BY started_at DESC) AS rn FROM {tname('meta','run_log')})
SELECT r.layer, r.object_id, o.data_owner, r.status, r.started_at, ROUND(r.duration_sec, 1) AS duration_sec,
       r.rows_read, r.rows_written, r.rows_quarantined, r.rows_deduplicated, r.message
FROM r LEFT JOIN {tname('meta','ingestion_object')} o ON r.object_id = o.object_id
WHERE rn = 1 ORDER BY CASE r.layer WHEN 'bronze' THEN 1 WHEN 'silver' THEN 2 ELSE 3 END, r.object_id"""))

### 2. Failures in the last 7 days + how to rerun

In [ ]:
fails = spark.sql(f"""SELECT started_at, layer, object_id, run_id, parent_run_id, message FROM {tname('meta','run_log')}
                      WHERE status = 'FAILED' AND started_at >= current_timestamp() - INTERVAL 7 DAYS ORDER BY started_at DESC""")
display(fails)
ids = sorted({r["object_id"] for r in fails.where("layer IN ('bronze','silver')").collect()})
if ids:
    print("Rerun options:")
    print(f"  * pipeline pl_mdf_orchestrator with rerun_failed_only = true")
    print(f"  * notebookutils.notebook.run('nb_mdf_10_landing_to_bronze', 1800, {{'object_ids': '{','.join(ids)}'}})")
    print(f"  * notebookutils.notebook.run('nb_mdf_20_bronze_to_silver', 1800, {{'object_ids': '{','.join(ids)}'}})")

### 3. Freshness vs SLA

In [ ]:
display(spark.sql(f"""
WITH last_ok AS (SELECT object_id, MAX(ended_at) AS last_success FROM {tname('meta','run_log')}
                 WHERE layer = 'silver' AND status = 'SUCCESS' GROUP BY object_id)
SELECT o.object_id, o.data_owner, o.sla_hours, l.last_success,
       ROUND((unix_timestamp(current_timestamp()) - unix_timestamp(l.last_success)) / 3600, 1) AS hours_since_success,
       CASE WHEN l.last_success IS NULL THEN 'NEVER LOADED'
            WHEN (unix_timestamp(current_timestamp()) - unix_timestamp(l.last_success)) / 3600 > o.sla_hours THEN 'SLA BREACH'
            ELSE 'OK' END AS freshness_status
FROM {tname('meta','ingestion_object')} o LEFT JOIN last_ok l ON o.object_id = l.object_id
WHERE o.active = true ORDER BY freshness_status, o.object_id"""))

### 4. Data-quality scorecard - technical vs business, latest run per rule

In [ ]:
display(spark.sql(f"""
WITH d AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY rule_id ORDER BY evaluated_at DESC) AS rn FROM {tname('meta','dq_result')})
SELECT d.rule_category, d.object_id, d.rule_id, d.rule_name, r.rule_owner, d.severity, d.rows_evaluated, d.rows_failed,
       ROUND(100 - d.failed_pct, 2) AS pass_pct, d.outcome
FROM d JOIN {tname('meta','dq_rule')} r ON d.rule_id = r.rule_id
WHERE d.rn = 1 ORDER BY d.rule_category, pass_pct"""))

In [ ]:
display(spark.sql(f"""
SELECT rule_category, COUNT(DISTINCT rule_id) AS rules, SUM(rows_evaluated) AS rows_evaluated, SUM(rows_failed) AS rows_failed,
       ROUND(100 - 100.0 * SUM(rows_failed) / NULLIF(SUM(rows_evaluated), 0), 3) AS pass_pct
FROM {tname('meta','dq_result')} GROUP BY rule_category"""))

### 5. Quarantine - open vs resolved (late-arriving rows resolve automatically)

In [ ]:
rows = []
for o in get_objects():
    q = tname("quarantine", o["silver_table"])
    if table_exists(q):
        for r in spark.table(q).groupBy("_quarantine_status").count().collect():
            rows.append((o["object_id"], q, r["_quarantine_status"], r["count"]))
if rows:
    display(spark.createDataFrame(rows, "object_id string, quarantine_table string, status string, rows long"))
else:
    print("Nothing in quarantine")

In [ ]:
# Drill into one quarantine table: which rules failed?
example = next((r[1] for r in rows), None) if rows else None
if example:
    display(spark.table(example).select(F.explode("_failed_rules").alias("rule_id"), "_quarantine_status")
            .groupBy("rule_id", "_quarantine_status").count().orderBy(F.desc("count")))

### 6. Watermarks (what the next run will pick up)

In [ ]:
display(spark.sql(f"""
WITH w AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY object_id, layer ORDER BY updated_at DESC) AS rn FROM {tname('meta','watermark')})
SELECT object_id, layer, watermark_value, updated_at FROM w WHERE rn = 1 ORDER BY object_id, layer"""))

### 7. SCD2 history example - how an entity (asset, store, ward, account ...) changed over time

In [ ]:
pk = get_pack()
name, key = tname("silver", pk["entity"]["table"]), pk["entity"]["id"]
if table_exists(name):
    multi = spark.table(name).groupBy(key).count().where("count > 1").limit(3)
    display(spark.table(name).join(multi.select(key), key)
            .select(key, "status", pk["entity"]["tier_col"], "_valid_from", "_valid_to", "_is_current").orderBy(key, "_valid_from"))
